# How do you make the final shipping decision?
**Topics:** Decision Matrix · Asymmetry Rule · When to Ship · When Not to Ship · Deployment Strategy · Post-Launch Review · Model Card

---
## Abbreviation Reference

| Abbreviation | Full Name |
|---|---|
| CI | Confidence Interval |
| DiD | Difference-in-Differences |
| IV | Instrumental Variables |
| MDE | Minimum Detectable Effect |
| ML | Machine Learning |
| PSM | Propensity Score Matching |
| RDD | Regression Discontinuity Design |
| ROI | Return on Investment |
| SRM | Sample Ratio Mismatch |

---
## 1. Combining all signals

### Input checklist before making a decision
- Offline evaluation result → see bd1_evaluating_results.ipynb Section 1
- A/B test or causal model result → see bd1_evaluating_results.ipynb Sections 2 and 3
- Cost-benefit and break-even → see bd2_cost_benefit.ipynb
- Risk and failure mode assessment (see Section 2 below)

### Decision matrix

| Signal | Weak | Moderate | Strong |
|---|---|---|---|
| Offline eval | Below MDE or CI includes zero | Above MDE, wide CI | Above MDE, tight CI, clean slices |
| A/B / causal result | SRM present, or non-significant, or pre-registered primary metric missed | Significant with caveats (short duration, novelty risk) | Significant, pre-registered, guardrails held, segments consistent |
| Cost-benefit | Below break-even | At break-even with wide uncertainty | Comfortably above break-even at lower CI bound |
| Risk | High irreversibility, no fallback | Moderate irreversibility, fallback exists | Reversible, rollback fast, fallback tested |

### Decision rule
- All four signals strong → full rollout
- Three strong, one moderate → canary rollout with tight guardrails
- Any signal weak → do not ship until resolved, or explicitly accept the risk in writing
- No single signal overrides the others — all four matter

---
## 2. The asymmetry rule

### Core principle
- Reversible failure → lower bar to ship; learn from production
- Irreversible failure → raise the evidence bar significantly before shipping

### Assessing reversibility

| Dimension | Low reversibility (raise the bar) | High reversibility (lower bar OK) |
|---|---|---|
| Rollback time | Hours to days | Minutes |
| Affected user scope | All users, no opt-out | Small segment, easy to exclude |
| Downstream dependencies | Other systems depend on model output | Self-contained, no downstream consumers |
| User trust impact | Affects core user experience or payment flow | Peripheral feature, low visibility |
| Legal / compliance | Regulated domain (credit, healthcare, hiring) | Unregulated, low stakes |
| Data side effects | Writes to production data, trains future models | Read-only, no feedback loops |

### Risk and failure mode checklist

| Risk | Questions to ask |
|---|---|
| Model failure modes | What happens when the model is confidently wrong? Is there a fallback? |
| Distribution shift | How will you detect when the model degrades over time? |
| Fairness and bias | Does performance vary unfairly across demographic groups? |
| Adversarial inputs | Can users game the model? (especially for ranking, fraud, spam) |
| Feedback loops | Does the model's output influence its own future training data? |

### Gotchas
- Regulated domains (credit scoring, hiring, healthcare) have legal irreversibility even if technical rollback is fast
- Feedback loops (model output → training data → next model) create compounding risk that is hard to reverse

---
## 3. When to ship

| Scenario | Decision | Rationale |
|---|---|---|
| All signals strong, reversible | Full rollout | Evidence is solid, downside is bounded |
| Mixed signals, reversible, cost low | Canary rollout with guardrails | Learn from production with limited exposure |
| Strong A/B but weak causal validation | Partial rollout to high-confidence segments | Limit exposure while gathering more evidence |
| Borderline lift, low cost, reversible | Ship with tight monitoring | The cost of delay may exceed the cost of a small miss |
| Strong offline, no A/B yet | Shadow mode first, then canary | Validate in production before real exposure |

### The cost of not shipping
- Delay has a cost: competitor advantage, missed revenue, opportunity cost of the team
- When failure is reversible and bounded, the cost of delay can exceed the cost of a cautious ship
- Document the expected cost of delay explicitly — it is often invisible in the decision

---
## 4. When not to ship

| Scenario | Reason | What to do instead |
|---|---|---|
| Borderline result + high irreversibility + no fallback | Risk of harm exceeds expected value | Gather more evidence, run a longer A/B, or reduce scope |
| Guardrail metric regressed | Known harm to an important signal | Investigate root cause before proceeding |
| Sample Ratio Mismatch (SRM) detected and unresolved | A/B result is invalid | Fix randomization, rerun the test |
| Causal assumption not credible | Estimate may be confounded | Strengthen evidence with additional methods or a targeted A/B |
| Break-even not met at lower CI bound | Expected value is negative at conservative estimate | Reduce cost, increase volume, or abandon |
| No clear decision-maker or rollback plan | Operational risk | Define ownership and rollback procedure before shipping |

### Gotchas
- "We've already built it" is not a reason to ship — sunk cost is irrelevant to the forward-looking decision
- Pressure to ship by a deadline should never override a failed guardrail metric

---
## 5. Deployment strategy pointer

### Choosing the right strategy

| Strategy | When to use | Key property |
|---|---|---|
| Shadow mode / dark launch | Confidence is low, risk is high | Model runs in parallel but output is not served — safe to compare without user exposure |
| Canary rollout | Default for most launches | Start at 1–5% of traffic, watch guardrails, increase gradually |
| Champion-challenger | Old model must stay live during validation | Route a fixed fraction to challenger permanently until confidence threshold is met |
| Full rollout | All signals strong, canary metrics stable | Only after canary phase — never skip directly to full rollout on first deploy |

### Auto-rollback triggers
- Define triggers before deploying — not after something goes wrong
- Common triggers: error rate increase > X%, latency p99 > Y ms, primary metric drop > Z%, guardrail regression
- Triggers should be automated where possible — human review loops add latency

### See also
- Implementation details for each strategy → ds7_deployment.ipynb

---
## 6. Post-launch review

### Review cadence

| Checkpoint | What to check | Action if degraded |
|---|---|---|
| 2-week review | Did the A/B lift hold in production? Did novelty effects decay? | Roll back if primary metric is below pre-defined floor |
| 90-day review | Distribution shift? Feature drift? Model decay? | Trigger retraining or investigation |
| Ongoing monitoring | Prediction distribution, feature distributions, business metrics | Alert on any guardrail breach |

### Monitoring checklist

| Signal | Why it matters |
|---|---|
| Prediction score distribution | Drift in score distribution signals input feature drift or population shift |
| Feature distributions | Individual feature drift helps diagnose which upstream data changed |
| Primary business metric | Ground truth of whether the model is delivering value |
| Guardrail metrics | Early warning system for unintended harm |
| Latency and error rate | Operational health — independent of model quality |

### Gotchas
- Set all alert thresholds before launch — retrospective threshold-setting is p-hacking for production
- Distribution shift is slow and silent — eyeballing dashboards misses it; use statistical drift tests like Population Stability Index (PSI action thresholds ≈ 0.1 moderate, ≈ 0.25 significant)

---
## 7. Model card documentation

### What it is
- A structured document capturing everything a future engineer or auditor needs to understand, operate, and maintain the model
- Required for governance — especially in regulated domains
- Complete before shipping, not after

### Model card checklist

| Section | What to document |
|---|---|
| Model overview | Name, version, purpose, intended use, out-of-scope uses |
| Training data | Data sources, date range, preprocessing steps, known biases or gaps |
| Evaluation results | Primary metric, test set description, Confidence Interval (CI), date of evaluation |
| Slice performance | Performance broken down by key segments — flag any gaps |
| Known failure modes | Conditions under which the model is known to underperform or fail |
| Fairness analysis | Performance parity across demographic groups; any disparate impact found |
| Deployment details | Serving mode, latency, infrastructure, rollback procedure |
| Retraining cadence | Trigger-based or scheduled? Who owns it? How long does a retraining run take? |
| Contacts and ownership | Model owner, on-call contact, escalation path |

### Gotchas
- A model card written after an incident is a post-mortem, not governance
- "Intended use" is as important as "known failure modes" — misuse is often the actual risk